In [2]:
import gradio as gr

class ChatBotUI:
    def __init__(self, chat_function, pdf_upload_function=None):
        self.chat_function = chat_function
        self.pdf_upload_function = pdf_upload_function
        self.chat_UI = self.init_chat_UI()

    def init_chat_UI(self):
        with gr.Blocks(title="My Chatbot") as chat_UI:
            gr.Markdown("# My Chatbot")

            chatbot = gr.Chatbot(label="ChatBot", height=500)

            # PDF Upload Row
            with gr.Row():
                pdf_upload = gr.File(
                    label="Upload PDF",
                    file_types=[".pdf"],
                    visible=True
                )
                upload_btn = gr.Button("📄 Upload PDF", variant="secondary")

            pdf_status = gr.Textbox(
                label="Upload Status",
                visible=False,
                interactive=False
            )

            # Message Input Row
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Type a message...",
                    label="You",
                    scale=9,
                    show_label=False
                )
                send_btn = gr.Button("Send ➤", variant="primary", scale=1)

            gr.Examples(
                examples=["Hello!", "Tell me a joke", "What can you do?"],
                inputs=msg_box
            )
            def respond(user_message, history):
                history = history or []
                bot_reply = self.chat_function(user_message, history)
                history.append((user_message, bot_reply))
                return "", history

            send_btn.click(
                fn=respond,
                inputs=[msg_box, chatbot],
                outputs=[msg_box, chatbot]
            )

            msg_box.submit(          # also allow pressing Enter
                fn=respond,
                inputs=[msg_box, chatbot],
                outputs=[msg_box, chatbot]
            )

            # ── Upload button logic ────────────────────────────────────
            def handle_upload(file):
                if file is None:
                    return gr.update(value="No file selected.", visible=True)

                if self.pdf_upload_function:
                    result = self.pdf_upload_function(file.name)
                    return gr.update(value=result, visible=True)

                return gr.update(
                    value=f"✅ '{file.name}' uploaded successfully!",
                    visible=True
                )

            upload_btn.click(
                fn=handle_upload,
                inputs=[pdf_upload],
                outputs=[pdf_status]
            )

        return chat_UI

    def launch_chat_UI(self):
        self.chat_UI.launch()

In [3]:
# With a PDF processor
def my_chat(message, history):
    return f"Echo: {message}"

def process_pdf(filepath):
    # your PDF processing logic here
    return f"✅ PDF processed: {filepath}"

ui = ChatBotUI(chat_function=my_chat, pdf_upload_function=process_pdf)
ui.launch_chat_UI()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
